# 🌊 瓦磘溝水質辨識 — YOLOv8 訓練

**三個水質等級：**
- ✅ `clean` — 乾淨（資料夾 5）
- ⚠️ `turbid` — 混濁（資料夾 3）
- ❌ `dirty` — 髒（資料夾 1）

### 使用步驟
1. 點選上方選單 **執行階段 → 變更執行階段類型 → T4 GPU**
2. 依序執行每個儲存格（Shift + Enter）

## Step 1 — 確認 GPU 並安裝套件

In [ ]:
# 確認 GPU
!nvidia-smi
import torch
print(f'\nPyTorch 版本：{torch.__version__}')
print(f'GPU 可用：{torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型號：{torch.cuda.get_device_name(0)}')

In [ ]:
# 安裝套件
!pip install ultralytics gdown -q
print('✅ 套件安裝完成')

## Step 2 — 掛載 Google Drive 並下載資料集

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Google Drive 已掛載')

In [ ]:
import gdown, shutil, random
from pathlib import Path

FOLDER_ID   = '1p4i4_PUK0FwDXd8C2JW5srdiZxH5uX8m'
RAW_DIR     = Path('/content/raw_data')
DATASET_DIR = Path('/content/dataset')
SPLIT_RATIO = 0.8
SEED        = 42

LABEL_MAP = {'1': 'dirty', '3': 'turbid', '5': 'clean'}
IMG_EXTS  = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

# 下載
if not RAW_DIR.exists():
    print('📥 下載中（約需 1–3 分鐘）...')
    gdown.download_folder(
        f'https://drive.google.com/drive/folders/{FOLDER_ID}',
        output=str(RAW_DIR), quiet=False, use_cookies=False
    )
else:
    print(f'ℹ️  已有 {RAW_DIR}，跳過下載')

# 顯示下載結果
print('\n下載後的資料夾結構：')
for d in sorted(RAW_DIR.iterdir()):
    if d.is_dir():
        imgs = list(d.rglob('*'))
        imgs = [f for f in imgs if f.suffix.lower() in IMG_EXTS]
        print(f'  {d.name}/  →  {len(imgs)} 張圖片')

In [ ]:
# 整理成 train / val 結構
random.seed(SEED)

if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split in ['train', 'val']:
    for label in LABEL_MAP.values():
        (DATASET_DIR / split / label).mkdir(parents=True, exist_ok=True)

stats = {}
for folder_name, label in LABEL_MAP.items():
    src_dir = RAW_DIR / folder_name
    if not src_dir.exists():
        # 模糊比對
        matches = [d for d in RAW_DIR.iterdir() if d.is_dir() and folder_name in d.name]
        if matches:
            src_dir = matches[0]
        else:
            print(f'✗ 找不到資料夾 {folder_name}')
            continue

    images = sorted([f for f in src_dir.rglob('*') if f.suffix.lower() in IMG_EXTS])
    random.shuffle(images)
    cut = int(len(images) * SPLIT_RATIO)

    for img in images[:cut]:
        shutil.copy(img, DATASET_DIR / 'train' / label / img.name)
    for img in images[cut:]:
        shutil.copy(img, DATASET_DIR / 'val' / label / img.name)

    stats[label] = {'train': cut, 'val': len(images) - cut, 'total': len(images)}
    print(f'✅ {label:10s}：訓練 {cut:3d} 張 / 驗證 {len(images)-cut:3d} 張')

total = sum(v['total'] for v in stats.values())
print(f'\n📊 共 {total} 張圖片，存於 {DATASET_DIR}')

## Step 3 — 訓練模型

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8s-cls.pt')  # small 模型，準確率與速度均衡

results = model.train(
    data     = str(DATASET_DIR),
    epochs   = 100,
    imgsz    = 224,
    batch    = 32,
    project  = '/content/runs',
    name     = 'water_quality',
    patience = 20,
    exist_ok = True,
)

print('\n🎉 訓練完成！')

## Step 4 — 查看訓練結果

In [ ]:
from IPython.display import Image, display
from pathlib import Path

result_dir = Path('/content/runs/water_quality')

# 顯示訓練曲線
for img_name in ['results.png', 'confusion_matrix.png', 'confusion_matrix_normalized.png']:
    img_path = result_dir / img_name
    if img_path.exists():
        print(f'\n📈 {img_name}')
        display(Image(str(img_path)))

## Step 5 — 推論測試

In [ ]:
from ultralytics import YOLO
from pathlib import Path
from IPython.display import Image, display

LABEL_ZH = {'clean': '✅ 乾淨', 'turbid': '⚠️ 混濁', 'dirty': '❌ 髒'}

model = YOLO('/content/runs/water_quality/weights/best.pt')

# 從 val 集隨機抽 6 張測試
import random
test_imgs = []
for label in ['clean', 'turbid', 'dirty']:
    imgs = list((DATASET_DIR / 'val' / label).glob('*'))
    if imgs:
        test_imgs.append(random.choice(imgs))

print('🔍 推論測試結果：')
print('─' * 50)
for img_path in test_imgs:
    true_label = img_path.parent.name
    results    = model(str(img_path))
    r          = results[0]
    pred       = r.names[r.probs.top1]
    conf       = r.probs.top1conf.item()
    correct    = '✓' if pred == true_label else '✗'

    print(f'{correct} 真實：{LABEL_ZH[true_label]:12s}  預測：{LABEL_ZH[pred]:12s}  信心度：{conf:.1%}')
    display(Image(str(img_path), width=200))

## Step 6 — 儲存模型到 Google Drive

In [ ]:
import shutil
from pathlib import Path

# 儲存到 Google Drive
SAVE_DIR = Path('/content/drive/MyDrive/water_quality_model')
SAVE_DIR.mkdir(parents=True, exist_ok=True)

src  = Path('/content/runs/water_quality/weights/best.pt')
dest = SAVE_DIR / 'best.pt'
shutil.copy(src, dest)

# 也複製訓練結果圖
for f in Path('/content/runs/water_quality').glob('*.png'):
    shutil.copy(f, SAVE_DIR / f.name)

print(f'✅ 模型已儲存到 Google Drive：{dest}')
print(f'📊 結果圖表也已複製過去')

# 顯示模型大小
size_mb = dest.stat().st_size / 1024 / 1024
print(f'📦 模型大小：{size_mb:.1f} MB')

## Step 7 — （選用）匯出 ONNX 格式
方便部署到 Vislab 或其他平台

In [ ]:
from ultralytics import YOLO

model = YOLO('/content/runs/water_quality/weights/best.pt')
model.export(format='onnx', imgsz=224)

# 複製到 Drive
import shutil
shutil.copy(
    '/content/runs/water_quality/weights/best.onnx',
    '/content/drive/MyDrive/water_quality_model/best.onnx'
)
print('✅ ONNX 模型已儲存到 Google Drive')